In [23]:
import pandas as pd
df = pd.read_excel('/content/Telco-Customer-Churn.csv')
print(df.columns.tolist())


['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


In [24]:
import pandas as pd
import numpy as np


df = pd.read_excel('/content/Telco-Customer-Churn.csv')

df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [26]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [27]:
df['Churn Label'].value_counts()

,count
Churn Label,
No,5174
Yes,1869


In [28]:
df['Total Charges'] = df['Total Charges'].fillna(0)

In [29]:
vazios = df['Total Charges'].isna().sum()
print(vazios)

0


#Segmentação de Clientes (Tiering), definir a categoria dos clientes

In [30]:
def classificar_cliente(valor):
    if valor < 40:
        return 'Starter'
    elif valor <= 80:
        return 'Pro'
    else:
        return 'Enterprise'

In [31]:
df['Customer_Tier'] = df['Monthly Charges'].apply(classificar_cliente)

In [32]:
contagem_tiers = df['Customer_Tier'].value_counts()
print(contagem_tiers)

Customer_Tier
Enterprise    2666
Pro           2540
Starter       1837
Name: count, dtype: int64


#


Usei o np.random.randint (gerar números aleatórios) simplesmente para fabricar este cenário. Criei dados falsos, mas com lógica verdadeira, isto porque o kagle não tem estes dados

In [33]:
import numpy as np

df['Ultimo_Login_Dias'] = np.where (
    df['Churn Label'] == 'Yes',
    np.random.randint(45, 91, size =len(df)),
    np.random.randint(1, 16, size=len(df))
)

df[['CustomerID', 'Churn Label', 'Ultimo_Login_Dias']].head(10)

,CustomerID,Churn Label,Ultimo_Login_Dias
0,3668-QPYBK,Yes,48
1,9237-HQITU,Yes,81
2,9305-CDSKC,Yes,70
3,7892-POOKP,Yes,68
4,0280-XJGEX,Yes,87
5,4190-MFLUW,Yes,89
6,8779-QRDMV,Yes,90
7,1066-JKSGK,Yes,66
8,6467-CHFZW,Yes,83
9,8665-UTDHZ,Yes,74


## Quanto cobramos aos clientes em media

In [34]:
round(df['Monthly Charges'].mean(), 2)

np.float64(64.76)

ARPU (Average Revenue Per User - Receita Média por Utilizador) é de 64.76€



Os clientes Enterprise (que pagam mais de 80€) estão a puxar esta média para cima e a financiar o negócio, enquanto os Starter estão abaixo.


#Taxa Churn

In [35]:
df['Churn Label'].value_counts(normalize=True)

,proportion
Churn Label,
No,0.73463
Yes,0.26537


Em negócios de subscrição e B2B, uma taxa de churn aceitável ronda os 3% a 5% ao ano, perder mais de um quarto da carteira de clientes (26.5%) é um cenário de alerta vermelho máximo.

#Taxa de cancelamento

In [36]:
df.groupby('Customer_Tier')['Churn Label'].value_counts(normalize=True)

Customer_Tier  Churn Label
Enterprise     No             0.660165
               Yes            0.339835
Pro            No             0.704724
               Yes            0.295276
Starter        No             0.884050
               Yes            0.115950
Name: proportion, dtype: float64

#Impacto Financeiro

In [37]:
df[ df['Churn Label'] == 'Yes' ]['Monthly Charges'].sum()

np.float64(139130.85)

estamos a falar de uma perda de receita de quase 140 mil euros todos os meses.

#Tech Support

In [38]:
df[ df['Tech Support'] == 'Yes' ]['Churn Label'].value_counts(normalize=True)

,proportion
Churn Label,
No,0.848337
Yes,0.151663


quando o cliente tem o Apoio Técnico ativado, a probabilidade de ele cancelar cai a pique para apenas 15.2% (os tais 84.8% de 'No' significam que eles ficam connosco). O serviço de Apoio Técnico está a segurar os clientes com unhas e dentes!

In [39]:
df[ df['Tech Support'] == 'No' ]['Churn Label'].value_counts(normalize=True)

,proportion
Churn Label,
No,0.583645
Yes,0.416355


Cliente COM Apoio Técnico: 15% de risco de fuga.
Cliente SEM Apoio Técnico: 42% de risco de fuga. (O risco quase triplica!)

#Análise de risco (Contrato mensal sem apoio tecnico)

In [40]:
df[
    (df['Contract'] == 'Month-to-month') &
    (df['Tech Support'] == 'No')
]['Churn Label'].value_counts(normalize=True)

,proportion
Churn Label,
Yes,0.503731
No,0.496269


Isto significa que se um cliente não tem fidelização (contrato mensal) e fica sem acesso a uma linha de Apoio Técnico, o risco é tão alto que atirar uma moeda ao ar tem a mesma probabilidade de prever se ele fica connosco ou se vai para a concorrência. É um autêntico balde furado. O Diretor de Vendas pode angariar os clientes que quiser, que neste cenário eles vão sair tão rápido quanto entram.


In [41]:
df.to_csv('B2B_Churn_Limpo2.csv', index=False)